# estimate E'

Self-contained batch tool (it does not use the strain module). Reads smoothed
mitral-annulus keypoints from a CSV, computes per-frame lengths from each keypoint to a
fixed apex anchor across a folder of segmented videos, saves annotated videos, and writes
E_Prime.csv. Point the three paths at your data.


In [ ]:
import cv2
import numpy as np
import math
import typing
import os
def show(frame):
    cv2.imshow("test", frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
def plot_point(frame,x,y,color=(0,255,0)):
    radius = 1
    thickness = -1
    return cv2.circle(frame, (x,y), radius, color, thickness)
def plot_line(frame,p1,p2,color=(0,191,255)):
    thickness = 2
    return cv2.line(frame, (p1[0],p1[1]), (p2[0],p2[1]), color, thickness)
def loadvideo(filename: str) -> np.ndarray:
    """Loads a video from a file.

    Args:
        filename (str): filename of video

    Returns:
        A np.ndarray with dimensions (channels=3, frames, height, width). The
        values will be uint8's ranging from 0 to 255.

    Raises:
        FileNotFoundError: Could not find `filename`
        ValueError: An error occurred while reading the video
    """

    if not os.path.exists(filename):
        raise FileNotFoundError(filename)
    capture = cv2.VideoCapture(filename)

    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

    v = np.zeros((frame_count, frame_height, frame_width, 3), np.uint8)

    for count in range(frame_count):
        ret, frame = capture.read()
        if not ret:
            raise ValueError("Failed to load frame #{} of {}.".format(count, filename))

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        v[count] = frame

    v = v.transpose((3, 0, 1, 2))

    return v
def savevideo(filename: str, array: np.ndarray, fps: typing.Union[float, int] = 1):
    """Saves a video to a file.

    Args:
        filename (str): filename of video
        array (np.ndarray): video of uint8's with shape (channels=3, frames, height, width)
        fps (float or int): frames per second

    Returns:
        None
    """

    c, f, height, width = array.shape

    if c != 3:
        raise ValueError("savevideo expects array of shape (channels=3, frames, height, width), got shape ({})".format(", ".join(map(str, array.shape))))
    fourcc = cv2.VideoWriter_fourcc('M', 'J', 'P', 'G')
    out = cv2.VideoWriter(filename, fourcc, fps, (width, height))

    for i in range(f):
        out.write(array[:, i, :, :].transpose((1, 2, 0)))

        


In [ ]:
import cv2
import numpy as np
def show(frame):
    cv2.imshow("test", frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
def load(vid):
    vid = loadvideo(vid)
    #vid = np.transpose(vid,(1,2,3,0))
    vid = vid[:,:,:,:112]
    return vid

In [ ]:
def midpoint(point1,point2):
    return (point1+point2)/2
def change(points):
    point_arr = [points[0]]
    for i in points[1:]:
        point_arr.append(midpoint(point_arr[-1],i))
    return np.array(point_arr)
import matplotlib.pyplot as plt
def delta(points):
    point_arr = [points[0]]
    for i in points[1:]:
        if np.linalg.norm(i-point_arr[-1])<2:
            point_arr.append(point_arr[-1])
        else:
            point_arr.append(i)
    return np.array(point_arr)
    #plt.hist(point_arr,bins=50)
    #plt.show()
    #for i in range(1,len(points)):
    #    print(np.linalg.norm(points[i]-points[i-1]))

In [ ]:
key_points = []
import pandas as pd
df = pd.read_csv('path/to/Smoothed_Bottom_Points.csv')

In [ ]:
df.head()

In [ ]:
from tqdm import tqdm
folder = 'path/to/segmented-videos'
output = 'path/to/output-folder'
filenames = []
frame_num = []
length1 = []
length2 = []

for vid in tqdm(os.listdir(folder)):
    try:
        if len(df[df.FileNames==vid])>0:
            temp = df[df.FileNames==vid]
            exists = True

        video_file = os.path.join(folder,vid)
        first = load(video_file)
        spare = first.copy()
        first = np.transpose(first,(1,2,3,0))
        video = []

        for i in range(0,len(first)):
            
            X1 = int(temp[temp.Frame ==i].X1.tolist()[0])
            Y1 = int(temp[temp.Frame ==i].Y1.tolist()[0])
            X2 = int(temp[temp.Frame ==i].X2.tolist()[0])
            Y2 = int(temp[temp.Frame ==i].Y2.tolist()[0])
            img = plot_point(first[i],X1,Y1)
            img = plot_line(img,[X1,Y1],[60,5])
            img = plot_point(img,X2,Y2)
            img = plot_line(img,[X2,Y2],[60,5])
            img = plot_point(img,60,5,color=(255,0,255))
            filenames.append(vid)
            frame_num.append(i)
            length1.append(np.sqrt((X1-60)**2+(Y1-5)**2))
            length2.append(np.sqrt((X2-60)**2+(Y2-5)**2))

            video.append(img)
        video = np.transpose(np.array(video),(3,0,1,2))

        savevideo(os.path.join(output,vid),video,fps=30)
    except:
        print(vid)


In [ ]:
plt.hist(length2)

In [ ]:
E_Prime = pd.DataFrame({"filenames":filenames,"Frame":frame_num,"Length 1":length1,"Length 2":length2})

In [ ]:
print(len(filenames),len(frame_num),len(length1),len(length2))

In [ ]:
E_Prime.to_csv("E_Prime.csv")